# Week 7 homework — Colony detection with YOLO

**Machine Learning for Bioinformatics**

This notebook implements homework items 2–4 from Practical 7, Part VI:

2. Validate image/label pairing and the YOLO format. Overlay ground-truth boxes on **at least three** images before training.
3. Train or evaluate **one YOLO nano** model and report precision, recall, mAP50 and mAP50-95 on a labelled validation split.
4. Use additional **CVAT** labels and retrain.

You already have labelled images in CVAT. Point `CVAT_EXPORT_PATH` at the YOLO 1.1 export folder or `.zip` (see section 8). The original course labels in `detection/colonies_annotation` are used for the first training run; the CVAT export is merged and the same nano model is retrained.

The pipeline follows Part IV of `Practical_07_CNN_CVAT_YOLO_Cellpose.ipynb`. We do **not** train the full YOLOv8/11/26 comparison.


## 0. Setup

Use a GPU if you can (`yolo11n`, 10 epochs, 640 px). On CPU this step is slower but still feasible because the labelled set is small (~185 train / ~48 val images).


In [ ]:
from pathlib import Path
import random
import re
import shutil
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

import torch
import torchvision
from ultralytics import YOLO

RANDOM_STATE = 221026
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

RUN_EXPENSIVE = True
SHOW_PLOTS = True
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}
YOLO_EPOCHS = 10
YOLO_IMGSZ = 640
YOLO_BATCH = 8
SELECTED_YOLO_NAME = 'YOLO11n'
SELECTED_YOLO_CHECKPOINT = 'yolo11n.pt'


def locate_week7_root():
    current = Path.cwd().resolve()
    for base in (current, *current.parents):
        for candidate in (base, base / 'week7'):
            if (candidate / 'detection').is_dir() and any(
                (candidate / name).is_dir() for name in ('segemntation', 'segmentation')
            ):
                return candidate
    return current


WEEK7_ROOT = locate_week7_root()
DETECTION_ROOT = WEEK7_ROOT / 'detection' / 'colonies_annotation'
YOLO_READY_ROOT = DETECTION_ROOT / 'ultralytics_ready'
DETECTION_TEST_ROOT = WEEK7_ROOT / 'detection' / 'test'
HOMETASK_ROOT = WEEK7_ROOT / 'hometask'

# Set this to your CVAT YOLO 1.1 export folder or zip file.
# Examples:
#   WEEK7_ROOT / 'cvat_export'
#   WEEK7_ROOT / 'cvat_export.zip'
#   Path('/mnt/c/Users/<you>/Downloads/cvat_export.zip')
CVAT_EXPORT_PATH = WEEK7_ROOT / 'cvat_export'

print('WEEK7_ROOT:', WEEK7_ROOT)
print('YOLO_READY_ROOT:', YOLO_READY_ROOT)
print('CVAT_EXPORT_PATH:', CVAT_EXPORT_PATH)


## 1. Image and annotation inventory

Classification answers *what is in the image*. Detection answers *what and where*, with one box per colony.

We inspect whether each image has a matching label file, and record basic imaging properties. These checks must be repeated after any CVAT export.


In [ ]:
def image_inventory_row(path, source, label_path=None):
    row = {
        'source': source,
        'file': path.name,
        'readable': False,
        'label_exists': label_path.exists() if label_path is not None else pd.NA,
    }
    try:
        with Image.open(path) as image:
            array = np.asarray(image)
            row.update({
                'readable': True,
                'format': image.format,
                'mode': image.mode,
                'width': image.width,
                'height': image.height,
                'dtype': str(array.dtype),
                'channels': 1 if array.ndim == 2 else array.shape[-1],
                'min_intensity': float(np.nanmin(array)),
                'max_intensity': float(np.nanmax(array)),
            })
    except Exception as error:
        row['error'] = repr(error)
    return row


def list_split_pairs(split):
    image_dir = YOLO_READY_ROOT / 'images' / split
    label_dir = YOLO_READY_ROOT / 'labels' / split
    if not image_dir.is_dir():
        return []
    pairs = []
    for image_path in sorted(image_dir.iterdir()):
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        pairs.append((image_path, label_dir / f'{image_path.stem}.txt'))
    return pairs


train_pairs = list_split_pairs('train')
val_pairs = list_split_pairs('val')
print('Prepared train pairs:', len(train_pairs))
print('Prepared val pairs:', len(val_pairs))

inventory_rows = []
for split, pairs in (('train', train_pairs), ('val', val_pairs)):
    for image_path, label_path in pairs[:5]:
        inventory_rows.append(image_inventory_row(image_path, f'ultralytics_{split}', label_path))
image_inventory = pd.DataFrame(inventory_rows)
display(image_inventory)

pairing = pd.DataFrame({
    'split': ['train'] * len(train_pairs) + ['val'] * len(val_pairs),
    'image': [p[0].name for p in train_pairs + val_pairs],
    'label_exists': [p[1].exists() for p in train_pairs + val_pairs],
})
print('Missing labels:', int((~pairing['label_exists']).sum()))
names_path = DETECTION_ROOT / 'obj.names'
DETECTION_CLASS_NAMES = names_path.read_text(encoding='utf-8').splitlines() if names_path.exists() else ['colony']
print('Class names:', DETECTION_CLASS_NAMES)


## 2. YOLO label format

Each non-empty annotation line has five values:

```text
class_id  x_center  y_center  width  height
```

The four box coordinates are normalized to `[0, 1]` relative to image width and height. This dataset has one class: `colony`. A parser can accept syntactically valid numbers while still using the wrong image, coordinate convention or class mapping, so we overlay boxes next.


In [ ]:
def read_yolo_label(label_path):
    records = []
    if not Path(label_path).exists():
        return records
    for line_number, line in enumerate(Path(label_path).read_text(encoding='utf-8').splitlines(), start=1):
        if not line.strip():
            continue
        values = line.split()
        if len(values) != 5:
            raise ValueError(f'{label_path}:{line_number} should have five values, got {len(values)}')
        class_id = int(values[0])
        coordinates = np.asarray([float(value) for value in values[1:]], dtype=float)
        if not np.all((coordinates >= 0) & (coordinates <= 1)):
            raise ValueError(f'Coordinates outside [0, 1] in {label_path}:{line_number}')
        records.append((class_id, *coordinates.tolist()))
    return records


annotation_summary = []
for split, pairs in (('train', train_pairs), ('val', val_pairs)):
    for image_path, label_path in pairs:
        labels = read_yolo_label(label_path) if label_path.exists() else []
        annotation_summary.append({
            'split': split,
            'image': image_path.name,
            'objects': len(labels),
            'empty': len(labels) == 0,
            'classes': sorted({row[0] for row in labels}),
        })
annotation_table = pd.DataFrame(annotation_summary)
display(annotation_table.groupby('split')['objects'].describe())
display(annotation_table.sort_values('objects', ascending=False).head(10))
print('Empty label files:', int(annotation_table['empty'].sum()) if len(annotation_table) else 0)

if SHOW_PLOTS and len(annotation_table):
    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    for axis, split in zip(axes, ('train', 'val')):
        subset = annotation_table[annotation_table['split'] == split]
        axis.hist(subset['objects'], bins=20, color='steelblue', edgecolor='white')
        axis.set_title(f'Colonies per image ({split})')
        axis.set_xlabel('objects')
    plt.tight_layout()
    plt.show()


## 3. Overlay ground-truth boxes on at least three images

This is homework item 2. We convert normalized YOLO boxes back to pixel coordinates and draw them on the raw images. Look for shifted boxes, missing touching colonies, border colonies and artifacts.


In [ ]:
def draw_ground_truth_boxes(axis, image_path, label_path):
    image = np.asarray(Image.open(image_path).convert('RGB'))
    height, width = image.shape[:2]
    axis.imshow(image)
    labels = read_yolo_label(label_path)
    for class_id, x_center, y_center, box_width, box_height in labels:
        x0 = (x_center - box_width / 2) * width
        y0 = (y_center - box_height / 2) * height
        axis.add_patch(plt.Rectangle(
            (x0, y0), box_width * width, box_height * height,
            fill=False, color='red', linewidth=1.5,
        ))
        class_name = DETECTION_CLASS_NAMES[class_id] if class_id < len(DETECTION_CLASS_NAMES) else str(class_id)
        axis.text(x0, max(0, y0 - 3), class_name, color='white', fontsize=8, backgroundcolor='red')
    axis.set_title(f'{image_path.name} | {len(labels)} objects')
    axis.axis('off')


overlay_pairs = [pair for pair in (train_pairs + val_pairs) if pair[1].exists()]
if overlay_pairs and SHOW_PLOTS:
    rng = np.random.default_rng(RANDOM_STATE)
    chosen = [overlay_pairs[i] for i in rng.choice(len(overlay_pairs), size=min(6, len(overlay_pairs)), replace=False)]
    n_rows = int(np.ceil(len(chosen) / 3))
    figure, axes = plt.subplots(n_rows, 3, figsize=(15, 5 * n_rows), squeeze=False)
    for axis in axes.ravel():
        axis.axis('off')
    for axis, (image_path, label_path) in zip(axes.ravel(), chosen):
        draw_ground_truth_boxes(axis, image_path, label_path)
    figure.suptitle('Ground-truth YOLO boxes', y=0.995)
    plt.tight_layout()
    plt.show()
    print('Overlaid images:')
    for image_path, _ in chosen:
        print('-', image_path.name)
else:
    print('No labelled images found for overlay.')


## 4. Write a local Ultralytics YAML

`colonies.yaml` may still contain a path from another machine. We rewrite it to this workspace before training.


In [ ]:
def write_colonies_yaml(root, yaml_path=None):
    yaml_path = Path(yaml_path) if yaml_path is not None else root / 'colonies.yaml'
    yaml_path.write_text(
        f'path: {root.resolve().as_posix()}\n'
        'train: images/train\n'
        'val: images/val\n'
        'names:\n  0: colony\n',
        encoding='utf-8',
    )
    return yaml_path


yolo_yaml = None
if train_pairs and val_pairs:
    yolo_yaml = write_colonies_yaml(YOLO_READY_ROOT)
    print(yolo_yaml.read_text())
else:
    print('ultralytics_ready is incomplete. Re-run the conversion cell in Practical 7 first.')


## 5. Train one YOLO nano model

Homework item 3 asks for a **nano** model. We use YOLO11-nano, 10 epochs, image size 640, batch 8, same protocol as the practical.


In [ ]:
def cuda_nms_available():
    if not torch.cuda.is_available():
        return False
    try:
        boxes = torch.tensor([[0.0, 0.0, 10.0, 10.0]], device='cuda')
        scores = torch.tensor([1.0], device='cuda')
        torchvision.ops.nms(boxes, scores, 0.5)
        return True
    except Exception as error:
        print('CUDA torchvision NMS unavailable:', repr(error))
        return False


YOLO_DEVICE = 0 if cuda_nms_available() else 'cpu'
YOLO_AMP = YOLO_DEVICE != 'cpu'
YOLO_PROJECT = WEEK7_ROOT / 'homework' / 'yolo_runs'
print('YOLO device:', YOLO_DEVICE)
print('Selected checkpoint:', SELECTED_YOLO_CHECKPOINT)


def metrics_row(name, metrics, checkpoint):
    return {
        'run': name,
        'checkpoint': str(checkpoint),
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'mAP50': float(metrics.box.map50),
        'mAP50-95': float(metrics.box.map),
    }


def train_yolo(run_name, data_yaml, checkpoint=SELECTED_YOLO_CHECKPOINT):
    detector = YOLO(checkpoint)
    detector.train(
        data=str(data_yaml),
        epochs=YOLO_EPOCHS,
        imgsz=YOLO_IMGSZ,
        batch=YOLO_BATCH,
        seed=RANDOM_STATE,
        device=YOLO_DEVICE,
        amp=YOLO_AMP,
        project=str(YOLO_PROJECT),
        name=run_name,
        exist_ok=True,
    )
    best_path = YOLO_PROJECT / run_name / 'weights' / 'best.pt'
    return best_path if best_path.exists() else None


baseline_best_path = YOLO_PROJECT / 'baseline_yolo11n' / 'weights' / 'best.pt'
if not baseline_best_path.exists():
    fallback = WEEK7_ROOT / 'best.pt'
    if fallback.exists():
        baseline_best_path = fallback

if RUN_EXPENSIVE and yolo_yaml is not None:
    trained = train_yolo('baseline_yolo11n', yolo_yaml)
    if trained is not None:
        baseline_best_path = trained
    print('Baseline checkpoint:', baseline_best_path)
else:
    print('Training skipped. Existing checkpoint will be used if present:', baseline_best_path)


## 6. Validation metrics

These numbers use the labelled validation split. They are not computed on `detection/test`, which has no ground truth until CVAT labels are added.


In [ ]:
baseline_model = None
metrics_table = pd.DataFrame()
if baseline_best_path.exists() and yolo_yaml is not None:
    baseline_model = YOLO(str(baseline_best_path))
    baseline_metrics = baseline_model.val(
        data=str(yolo_yaml), split='val', imgsz=YOLO_IMGSZ,
        device=YOLO_DEVICE, verbose=False,
    )
    metrics_table = pd.DataFrame([metrics_row('baseline (course labels)', baseline_metrics, baseline_best_path)])
    display(metrics_table)
else:
    print('Evaluation skipped because best.pt or the YAML is missing.')


## 7. Visual prediction check and error types

Homework question 3: are errors missed colonies (false negatives), extra boxes (false positives), or inaccurate boundaries?

Each row shows ground truth vs the nano-model prediction on the same validation image.


In [ ]:
def boxes_from_result(result):
    if result.boxes is None or len(result.boxes) == 0:
        return np.empty((0, 4)), np.empty(0, dtype=int), np.empty(0)
    return (
        result.boxes.xyxy.detach().cpu().numpy(),
        result.boxes.cls.detach().cpu().numpy().astype(int),
        result.boxes.conf.detach().cpu().numpy(),
    )


def draw_model_boxes(axis, image, boxes, classes, confidences, names):
    axis.imshow(image)
    for box, class_id, confidence in zip(boxes, classes, confidences):
        x0, y0, x1, y1 = box
        axis.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color='lime', linewidth=1.5))
        class_name = names.get(int(class_id), str(class_id)) if isinstance(names, dict) else str(class_id)
        axis.text(x0, max(0, y0 - 3), f'{class_name} {confidence:.2f}', color='black', fontsize=8, backgroundcolor='lime')
    axis.axis('off')


if baseline_model is not None and val_pairs:
    rng = np.random.default_rng(RANDOM_STATE)
    selected = [val_pairs[i] for i in rng.choice(len(val_pairs), size=min(3, len(val_pairs)), replace=False)]
    if SHOW_PLOTS:
        figure, axes = plt.subplots(len(selected), 2, figsize=(14, 5 * len(selected)), squeeze=False)
        for row, (image_path, label_path) in enumerate(selected):
            image = np.asarray(Image.open(image_path).convert('RGB'))
            result = baseline_model.predict(
                image, imgsz=YOLO_IMGSZ, conf=0.25, device=YOLO_DEVICE, verbose=False,
            )[0]
            boxes, classes, confidences = boxes_from_result(result)
            draw_ground_truth_boxes(axes[row, 0], image_path, label_path)
            axes[row, 0].set_title(f'Ground truth: {image_path.name}')
            draw_model_boxes(axes[row, 1], image, boxes, classes, confidences, baseline_model.names)
            axes[row, 1].set_title(f'Prediction: {len(boxes)} boxes')
        figure.suptitle('Validation: ground truth vs YOLO11n', y=1.0)
        plt.tight_layout()
        plt.show()
else:
    print('Prediction preview skipped.')


## 8. Uncertainty ranking on unlabeled images

`detection/test` has no labels. We score each image by low confidence and disagreement under a horizontal flip. High-ranking images should look difficult to a human annotator (homework question 4). This is **not** a ground-truth error rate.


In [ ]:
def box_iou(box_a, box_b):
    left, top = max(box_a[0], box_b[0]), max(box_a[1], box_b[1])
    right, bottom = min(box_a[2], box_b[2]), min(box_a[3], box_b[3])
    intersection = max(0.0, right - left) * max(0.0, bottom - top)
    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


def flip_boxes_back(boxes, image_width):
    restored = boxes.copy()
    if len(restored):
        restored[:, 0] = image_width - boxes[:, 2]
        restored[:, 2] = image_width - boxes[:, 0]
    return restored


def matched_tta_iou(boxes_a, classes_a, boxes_b, classes_b):
    if len(boxes_a) == 0 and len(boxes_b) == 0:
        return 1.0
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return 0.0
    used_b, pair_ious = set(), []
    for index_a, class_a in enumerate(classes_a):
        candidates = [
            index_b for index_b, class_b in enumerate(classes_b)
            if index_b not in used_b and class_b == class_a
        ]
        if candidates:
            index_b = max(candidates, key=lambda j: box_iou(boxes_a[index_a], boxes_b[j]))
            used_b.add(index_b)
            pair_ious.append(box_iou(boxes_a[index_a], boxes_b[index_b]))
        else:
            pair_ious.append(0.0)
    pair_ious.extend([0.0] * (len(boxes_b) - len(used_b)))
    return float(np.mean(pair_ious)) if pair_ious else 0.0


test_images = sorted(
    path for path in DETECTION_TEST_ROOT.iterdir()
    if path.suffix.lower() in IMAGE_EXTENSIONS
) if DETECTION_TEST_ROOT.is_dir() else []
print('Unlabeled test images:', len(test_images))

uncertainty_table = pd.DataFrame()
prediction_cache = {}
if baseline_model is not None and test_images:
    uncertainty_rows = []
    for image_path in test_images:
        image = np.asarray(Image.open(image_path).convert('RGB'))
        flipped_image = np.ascontiguousarray(image[:, ::-1, :])
        original_result = baseline_model.predict(image, imgsz=YOLO_IMGSZ, conf=0.25, device=YOLO_DEVICE, verbose=False)[0]
        flipped_result = baseline_model.predict(flipped_image, imgsz=YOLO_IMGSZ, conf=0.25, device=YOLO_DEVICE, verbose=False)[0]
        boxes_a, classes_a, confidences_a = boxes_from_result(original_result)
        boxes_b, classes_b, _ = boxes_from_result(flipped_result)
        boxes_b = flip_boxes_back(boxes_b, image.shape[1])
        flip_iou = matched_tta_iou(boxes_a, classes_a, boxes_b, classes_b)
        mean_confidence = float(np.mean(confidences_a)) if len(confidences_a) else 0.0
        count_difference = abs(len(boxes_a) - len(boxes_b)) / max(1, max(len(boxes_a), len(boxes_b)))
        uncertainty = 0.5 * (1.0 - mean_confidence) + 0.4 * (1.0 - flip_iou) + 0.1 * count_difference
        uncertainty_rows.append({
            'image': image_path.name,
            'path': str(image_path),
            'predictions': len(boxes_a),
            'mean_confidence': mean_confidence,
            'flip_iou': flip_iou,
            'uncertainty': uncertainty,
        })
        prediction_cache[str(image_path)] = (image, boxes_a, classes_a, confidences_a)
    uncertainty_table = pd.DataFrame(uncertainty_rows).sort_values('uncertainty', ascending=False).reset_index(drop=True)
    display(uncertainty_table.head(10))
    if SHOW_PLOTS and len(uncertainty_table):
        top_rows = uncertainty_table.head(6)
        figure, axes = plt.subplots(2, 3, figsize=(15, 9), squeeze=False)
        for axis in axes.ravel():
            axis.axis('off')
        for axis, row in zip(axes.ravel(), top_rows.itertuples(index=False)):
            image, boxes, classes, confidences = prediction_cache[row.path]
            draw_model_boxes(axis, image, boxes, classes, confidences, baseline_model.names)
            axis.set_title(f'{row.image}\nuncert={row.uncertainty:.3f} | flip IoU={row.flip_iou:.3f}')
        figure.suptitle('Highest-uncertainty unlabeled images', y=0.995)
        plt.tight_layout()
        plt.show()
else:
    print('Uncertainty ranking skipped.')


## 9. Import your CVAT labels and retrain

Homework item 4. Export the task from CVAT as **YOLO 1.1** and set `CVAT_EXPORT_PATH` to that folder or zip.

Typical CVAT YOLO 1.1 layout:

```text
obj.names
obj.data
train.txt
obj_train_data/
    image.jpg
    image.txt
```

The extra labelled images are copied into a new Ultralytics dataset that **keeps the original validation split unchanged**. That way the metric comparison is not confounded by a different val set. Matching image files are searched in the export itself, `hometask/`, and `detection/test/`.


In [ ]:
def unpack_cvat_export(export_path, extract_root):
    export_path = Path(export_path)
    if export_path.is_dir():
        return export_path
    if export_path.is_file() and export_path.suffix.lower() == '.zip':
        extract_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(export_path) as archive:
            archive.extractall(extract_root)
        return extract_root
    return None


def find_cvat_label_files(root):
    return [
        path for path in Path(root).rglob('*.txt')
        if path.name.lower() not in {'train.txt', 'obj.names', 'obj.data'}
        and not path.name.endswith('.yaml')
    ]


def find_image_for_stem(stem, search_roots):
    for root in search_roots:
        if root is None or not Path(root).exists():
            continue
        root = Path(root)
        for path in root.rglob('*'):
            if path.suffix.lower() in IMAGE_EXTENSIONS and path.stem == stem:
                return path
    return None


cvat_search_candidates = [
    CVAT_EXPORT_PATH,
    WEEK7_ROOT / 'cvat_export',
    WEEK7_ROOT / 'cvat_export.zip',
    HOMETASK_ROOT,
]
cvat_root = None
for candidate in cvat_search_candidates:
    unpacked = unpack_cvat_export(candidate, WEEK7_ROOT / 'homework' / 'cvat_unpacked') if Path(candidate).exists() else None
    if unpacked is None:
        continue
    if find_cvat_label_files(unpacked):
        cvat_root = unpacked
        break

print('CVAT root:', cvat_root)
if cvat_root is None:
    print(
        'No CVAT YOLO export found. Export the task from CVAT as YOLO 1.1, '
        'then set CVAT_EXPORT_PATH to that folder or zip and rerun this cell.'
    )


In [ ]:
COMBINED_ROOT = WEEK7_ROOT / 'homework' / 'yolo_combined'
combined_yaml = None
cvat_added = pd.DataFrame()

if cvat_root is not None and yolo_yaml is not None:
    if COMBINED_ROOT.exists():
        shutil.rmtree(COMBINED_ROOT)
    for split in ('train', 'val'):
        (COMBINED_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
        (COMBINED_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)
        for image_path, label_path in (train_pairs if split == 'train' else val_pairs):
            shutil.copy2(image_path, COMBINED_ROOT / 'images' / split / image_path.name)
            if label_path.exists():
                shutil.copy2(label_path, COMBINED_ROOT / 'labels' / split / f'{image_path.stem}.txt')

    existing_stems = {path.stem for path in (COMBINED_ROOT / 'images').rglob('*') if path.suffix.lower() in IMAGE_EXTENSIONS}
    val_stems = {path.stem for path, _ in val_pairs}
    search_roots = [cvat_root, HOMETASK_ROOT, DETECTION_TEST_ROOT, DETECTION_ROOT / 'obj_train_data']
    added_rows = []
    for label_path in find_cvat_label_files(cvat_root):
        try:
            labels = read_yolo_label(label_path)
        except Exception as error:
            print('Skipping invalid label', label_path, error)
            continue
        image_path = find_image_for_stem(label_path.stem, search_roots)
        if image_path is None:
            sibling = next(
                (
                    candidate for candidate in label_path.parent.iterdir()
                    if candidate.stem == label_path.stem and candidate.suffix.lower() in IMAGE_EXTENSIONS
                ),
                None,
            )
            image_path = sibling
        if image_path is None:
            print('No image for label', label_path.name)
            continue
        if label_path.stem in val_stems:
            # Keep the original validation split frozen.
            continue
        if image_path.stem in existing_stems:
            continue
        clean_stem = re.sub(r'[^A-Za-z0-9_.-]+', '_', image_path.stem)
        output_name = f'cvat_{clean_stem}{image_path.suffix.lower()}'
        shutil.copy2(image_path, COMBINED_ROOT / 'images' / 'train' / output_name)
        shutil.copy2(label_path, COMBINED_ROOT / 'labels' / 'train' / f'{Path(output_name).stem}.txt')
        added_rows.append({'image': output_name, 'objects': len(labels), 'source': str(image_path)})
        existing_stems.add(image_path.stem)
    cvat_added = pd.DataFrame(added_rows)
    combined_yaml = write_colonies_yaml(COMBINED_ROOT)
    print('Added CVAT-labelled training images:', len(cvat_added))
    display(cvat_added.head(20) if len(cvat_added) else cvat_added)
    print(combined_yaml.read_text())
    if SHOW_PLOTS and len(cvat_added):
        preview = list((COMBINED_ROOT / 'images' / 'train').glob('cvat_*'))[:3]
        figure, axes = plt.subplots(1, len(preview), figsize=(5 * max(1, len(preview)), 4), squeeze=False)
        for axis, image_path in zip(axes.ravel(), preview):
            label_path = COMBINED_ROOT / 'labels' / 'train' / f'{image_path.stem}.txt'
            draw_ground_truth_boxes(axis, image_path, label_path)
        figure.suptitle('Newly added CVAT annotations')
        plt.tight_layout()
        plt.show()
else:
    print('CVAT merge skipped.')


## 10. Retrain YOLO11n on course labels + CVAT labels

The validation split is the original one, so precision / recall / mAP changes come from the extra training labels, not from a different test set.


In [ ]:
retrained_best_path = None
if RUN_EXPENSIVE and combined_yaml is not None and cvat_root is not None:
    retrained_best_path = train_yolo('retrained_yolo11n_cvat', combined_yaml)
    print('Retrained checkpoint:', retrained_best_path)
    if retrained_best_path is not None:
        retrained_model = YOLO(str(retrained_best_path))
        retrained_metrics = retrained_model.val(
            data=str(combined_yaml), split='val', imgsz=YOLO_IMGSZ,
            device=YOLO_DEVICE, verbose=False,
        )
        metrics_table = pd.concat(
            [metrics_table, pd.DataFrame([metrics_row('retrained (+ CVAT labels)', retrained_metrics, retrained_best_path)])],
            ignore_index=True,
        )
        display(metrics_table)
else:
    print('Retraining skipped. Import CVAT labels first, then set RUN_EXPENSIVE=True.')

if len(metrics_table):
    metrics_table.to_csv(WEEK7_ROOT / 'homework' / 'detection_metrics.csv', index=False)
    print('Saved:', WEEK7_ROOT / 'homework' / 'detection_metrics.csv')


## Report answers (detection)

Fill these after training. They map to homework questions 1, 3, 4 and 6.

**1. What image, annotation, and split checks were necessary before modelling?**

- Image/label pairing, YOLO five-value format, coordinates in `[0, 1]`, visual overlay of at least three images, frozen validation split when adding CVAT labels.

**3. What are the most common YOLO errors: missed objects, false detections, or inaccurate box boundaries?**

- After looking at the validation overlays: ...

**4. Does the uncertainty ranking select images that look difficult or ambiguous to a human annotator?**

- Yes/no, with examples: crowding, glare, tiny colonies, plate edges, ...

**6. What information would be needed to compare detection fairly across biological experiments?**

- A written annotation protocol (border / touching / artifact rules), the same camera and plate conditions, and evaluation on a held-out labelled set rather than on the images that were just added from CVAT.
